In [ ]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [ ]:
# Read the supplied ticker file. It may have a header or just one column of ticker symbols.
ticker_data = pd.read_csv("Tickers.csv", header=None)
tickers = ticker_data.iloc[:, 0].dropna().astype(str).str.strip()

if not tickers.empty and tickers.iloc[0].lower() in ["ticker", "symbol"]:
    tickers = tickers.iloc[1:]

tickers = tickers[tickers != ""].drop_duplicates()

raw_info = {}
for ticker in tickers:
    try:
        raw_info[ticker] = yf.Ticker(ticker).info
    except Exception:
        raw_info[ticker] = {}

In [ ]:
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = raw_info.get(ticker)
        type_s.append(tick.get('quoteType'))
        currency.append(tick.get('currency'))
        market.append(tick.get('market'))
        sector.append(tick.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

check if it is eligible

In [ ]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])

eligible_stocks = all_stocks[mask & mask1 & mask2]
eligible_stocks

check for volume

In [ ]:
# Get data from the volume window through the purchase date.
buy_date = "2026-11-27"
global_start = "2025-10-01"
global_end = (pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

tickers_list = eligible_stocks.index.tolist().copy()
hist_data = yf.download(
    tickers_list,
    start=global_start,
    end=global_end,
    group_by="ticker"
)

In [ ]:
start='2025-10-01'
end='2026-10-01'
min_avg_volume=5000

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[start:end]
    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history.Volume.mean()

mask = eligible_stocks['Avg_Volume']>=min_avg_volume
eligible_stocks = eligible_stocks[mask]

eligible_stocks

take the closing price

In [ ]:
end_date = (pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[buy_date:end_date]
    if not history.empty:
        eligible_stocks.loc[ticker, "Close"] = history.loc[buy_date, "Close"]

eligible_stocks

convert to CAD

In [ ]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = usdcad.loc[buy_date, 'Close']
for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

fee application function

PORTFOLIO A

Estimate beta against the S&P 500 over the year before the purchase date.

In [ ]:
# Estimate beta manually using the year before the purchase date and the S&P 500 as the market benchmark.
beta_start = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
market_hist = yf.download("^GSPC", start=beta_start, end=end_date)
market_close = market_hist["Close"]["^GSPC"]
market_returns = market_close.pct_change()

for ticker in eligible_stocks.index.tolist():
    stock_close = hist_data[ticker]["Close"].loc[beta_start:buy_date]
    stock_returns = stock_close.pct_change(fill_method=None)

    return_data = pd.concat(
        [stock_returns, market_returns],
        axis=1,
        join="inner"
    ).dropna()

    covariance = return_data.iloc[:, 0].cov(return_data.iloc[:, 1])
    variance = return_data.iloc[:, 1].var()

    if variance != 0:
        eligible_stocks.loc[ticker, "Beta"] = covariance / variance

eligible_stocks

If we couldn't calculate a beta ourselves, use Yahoo's beta instead. If Yahoo doesn't have one either, just assume average risk (1.0)

In [ ]:
for ticker in eligible_stocks.index.tolist():
    beta = eligible_stocks.loc[ticker, 'Beta']

    if pd.isna(beta):
        yahoo_beta = raw_info[ticker].get('beta')
        if yahoo_beta is None:
            beta = 1.0
        else:
            beta = yahoo_beta
        eligible_stocks.loc[ticker, 'Beta'] = beta

eligible_stocks

1 year returns

In [ ]:
one_year_ago = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker]
    if one_year_ago in history.index and buy_date in history.index:
        old_price = history.loc[one_year_ago, 'Close']
        new_price = history.loc[buy_date, 'Close']
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        old_price = history['Close'].dropna().asof(pd.Timestamp(one_year_ago))
        new_price = history['Close'].dropna().asof(pd.Timestamp(buy_date))

    if pd.notna(old_price) and pd.notna(new_price):
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        eligible_stocks.loc[ticker, 'Return %'] = None

# Drop any stock we couldn't compute a 1-year return for (missing data that far back)
eligible_stocks = eligible_stocks.dropna(subset=['Return %'])
eligible_stocks

Combine Beta and Return into one Score

In [ ]:
beta_norm = (eligible_stocks['Beta'] - eligible_stocks['Beta'].min()) / (eligible_stocks['Beta'].max() - eligible_stocks['Beta'].min())
return_norm = (eligible_stocks['Return %'] - eligible_stocks['Return %'].min()) / (eligible_stocks['Return %'].max() - eligible_stocks['Return %'].min())

eligible_stocks['Score'] = (beta_norm + return_norm) / 2
eligible_stocks.sort_values('Score', ascending=False)

Split into a growth bucket (Technology + Communication Services, target 40%) and everything else (target 60%). If there aren't enough growth stocks, use fewer and lower the target so we never break the 15% per-stock cap

In [ ]:
#select sectors the portfolio is going to focus on
growth_sectors = ['Technology', 'Communication Services']

mask = eligible_stocks['Sector'].isin(growth_sectors)
growth_stocks = eligible_stocks[mask].sort_values('Score', ascending=False)
other_stocks = eligible_stocks[~mask].sort_values('Score', ascending=False)

# total stock count
n_total = 15
# use 5 selected sector stocks, or fewer if the ticker file doesn't have 5
n_growth = min(5, len(growth_stocks))   
n_other = n_total - n_growth

#take the n_growth number of the best stocks in the selected sectors
selected_growth = growth_stocks.head(n_growth).copy()
selected_other = other_stocks.head(n_other).copy()

# Quick sanity check: make sure no OTHER single sector is secretly dominating this bucket
selected_other['Sector'].value_counts()

Weight each bucket by Score, then cap any single stock at 15% and hand the extra weight to the rest of its bucket

In [ ]:
def cap_weights(df, cap=0.15):
    # add 0.0001 to prevent infinite loops due to rounding
    while (df['Weight'] > cap + 0.0001).any():
        excess = 0
        for ticker in df.index:
            if df.loc[ticker, 'Weight'] > cap:
                excess = excess + (df.loc[ticker, 'Weight'] - cap)
                df.loc[ticker, 'Weight'] = cap
    
        not_capped = df[df['Weight'] < cap]
        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

    return df
    
# can't ask for more than the 15% cap allows with this many stocks
target_growth = min(0.40, n_growth * 0.15)
target_other = 1 - target_growth

selected_growth['Weight'] = selected_growth['Score'] / selected_growth['Score'].sum() * target_growth
selected_other['Weight'] = selected_other['Score'] / selected_other['Score'].sum() * target_other

selected_growth = cap_weights(selected_growth)
selected_other = cap_weights(selected_other)

portfolio_A = pd.concat([selected_growth, selected_other])

print('Total weight:', portfolio_A['Weight'].sum())
print('Growth sector weight:', selected_growth['Weight'].sum())
portfolio_A[['Sector', 'Beta', 'Return %', 'Score', 'Weight']].sort_values('Weight', ascending=False)

In [ ]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

In [ ]:
budget = 1000000
def buy_stocks(portfolio, eligible):
    # first pass: split the budget by weight, ignoring fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * budget
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # add up what the fees would come out to with these share counts
    total_fees = 0
    for ticker in portfolio.index.tolist():
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        total_fees = total_fees + fee

    # second pass: shrink the budget by the fees, then size shares again
    investable = budget - total_fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * investable
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # final value + fee per stock, and a check that we spent close to budget
    total_value = 0
    total_fees_final = 0
    for ticker in portfolio.index.tolist():
        value = portfolio.loc[ticker, 'Shares'] * eligible.loc[ticker, 'Close_CAD']
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        portfolio.loc[ticker, 'Value'] = value
        portfolio.loc[ticker, 'Fee'] = fee
        total_value = total_value + value
        total_fees_final = total_fees_final + fee

    print('Total spent:', total_value + total_fees_final)
    print('Budget:', budget)
    print('Leftover:', budget - total_value - total_fees_final)

    return portfolio

portfolio_A = buy_stocks(portfolio_A, eligible_stocks)
portfolio_A

In [ ]:
Portfolio_Final_A_data = {
    'Ticker' : portfolio_A.index,
    'Price' : portfolio_A.Close.values,
    'Currency' : portfolio_A.Currency.values,
    'Shares' : portfolio_A.Shares.values,
    'Value' : portfolio_A.Value.values,
    # to reduce fees from the total value of the invested shares
    'Weight' : (portfolio_A.Value / portfolio_A.Value.sum()).values
}

Portfolio_Final_A = pd.DataFrame(Portfolio_Final_A_data, index = range(1, len(portfolio_A) + 1))
Portfolio_Final_A

In [ ]:
# Compare the current weights with an equal-weight version of the same 15 stocks.

fx_prices = yf.Ticker("USDCAD=X").history(
    start=one_year_ago,
    end=(pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
)["Close"]

fx_prices.index = pd.to_datetime(fx_prices.index).tz_localize(None)
price_history_cad = pd.DataFrame()

for ticker in portfolio_A.index:
    stock_prices = hist_data[ticker]["Close"].loc[one_year_ago:buy_date].copy()
    stock_prices.index = pd.to_datetime(stock_prices.index).tz_localize(None)

    if portfolio_A.loc[ticker, "Currency"] == "USD":
        fx_for_stock = fx_prices.reindex(stock_prices.index).ffill()
        stock_prices = stock_prices * fx_for_stock

    price_history_cad[ticker] = stock_prices

price_history_cad = price_history_cad.dropna()
growth_of_1_dollar = price_history_cad / price_history_cad.iloc[0]

score_weighted_value = pd.Series(0.0, index=growth_of_1_dollar.index)
equal_weight_value = pd.Series(0.0, index=growth_of_1_dollar.index)

for ticker in portfolio_A.index:
    score_weighted_value = score_weighted_value + (
        growth_of_1_dollar[ticker] * portfolio_A.loc[ticker, "Weight"]
    )
    equal_weight_value = equal_weight_value + (
        growth_of_1_dollar[ticker] / len(portfolio_A)
    )

score_weighted_returns = score_weighted_value.pct_change().dropna()
equal_weight_returns = equal_weight_value.pct_change().dropna()

comparison_A = pd.DataFrame(
    {
        "Historical price return": [
            score_weighted_value.iloc[-1] - 1,
            equal_weight_value.iloc[-1] - 1,
        ],
        "Daily CAD volatility": [
            score_weighted_returns.std(),
            equal_weight_returns.std(),
        ],
    },
    index=["Portfolio A weights", "Same stocks, equal weight"],
)

display(comparison_A * 100)
print("Comparison period:", price_history_cad.index[0].date(), "to", price_history_cad.index[-1].date())


# Final Explanations — Portfolio A

### Explanation 1 — Why Portfolio A makes financial sense

We designed Portfolio A for a client who wants growth and is comfortable taking on more risk. We score each eligible stock using its previous-year return and beta. We scale both measures to the same range, then average them so neither dominates just because it uses a different scale. A higher score can come from stronger past return, higher beta, or both.

We manually estimate beta using about one year of daily stock returns and S&P 500 returns. Beta measures how sensitive a stock has been to movements in the overall market. A higher beta can mean larger gains when the market rises, but also larger losses when it falls. In finance, investors generally expect to be compensated for accepting more systematic risk, but beta itself does not guarantee a higher return. We use a recent one-year estimate because the risk profile of fast-changing technology companies may differ from its five-year history. The shorter estimate can also be less stable. If the manual calculation is unavailable, our code falls back to Yahoo Finance’s beta and then to 1.0 if no value is available.

We select the highest-scoring stocks, with a target 40% of the portfolio in Technology and Communication Services and 60% in other sectors. Within each group, we give higher-scoring stocks more weight and cap each holding at 15%. This is score-based portfolio weighting: it gives more of the budget to stocks that better match our growth signals while limiting the impact of one company.

### Explanation 2 — Evidence and equal-weight comparison

We compare our score-weighted portfolio with an equal-weight portfolio holding the same selected stocks over the same dates. Both portfolios use consistent CAD values, with USD holdings converted using the exchange rate. The comparison table reports each portfolio’s historical price return and daily CAD volatility. Dividends are excluded, as required.

Because the holdings and time period are the same, the comparison helps us isolate the effect of our weighting choice. We use the return to assess historical growth and the volatility to assess how much daily portfolio returns moved around. We interpret the results together: a higher return with higher volatility means our weighting delivered more historical growth alongside larger fluctuations; a higher return with similar or lower volatility would be stronger evidence for our weighting over this period.

The stocks are ranked using return and beta from the same previous-year period used in the comparison. We therefore treat the results as historical evidence, not an independent test or a forecast.

### Explanation 3 — What could go wrong?

Our strategy relies on past return and beta, but neither is guaranteed to continue. A one-year beta may reflect recent market conditions better than a five-year beta, but a shorter period can also be less stable and more affected by unusual events. If technology stocks become less sensitive to the market or their past returns reverse, their scores may no longer indicate future performance.

A high-beta portfolio may lose more than the market during a downturn. The 15% limit reduces the impact of one holding, but several stocks can fall at the same time. Our score also does not assess company valuation, debt, or earnings quality. The 60% allocated outside the growth-sector group could still become concentrated in one industry, and exchange-rate changes can affect the CAD value of US holdings.